<a href="https://colab.research.google.com/github/Irtisam99/Deep_Learning/blob/main/Multilayer_Perceptron_(MNIST)_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

In [2]:
""" Deep Neural Network / Artificial Neural Network
1. Feed Forwad Neural Network  (Fully Connected normally)
    Example: Input Layer> Hidden Layer > Output Layer
2. Convulational Neural Network
    Example: Input layer (Image) > CNN layer (s) > Fully Connected Layer(s) > output layer
            Image (300 x 300)
            Raw features: 90000 pixels. Each pixel is a feature.
            CNN layers extracts feature maps from the image
3. Recurrent Neural Network: Sequential Data uses forward feed connection,The output of a neuron is looped back into itself as part of the input for the next step in the sequence.
      Hi, His name is Irtisam. He studies computer science. He ___
"""
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


cuda


In [3]:
""" Image -> class

Image: A 2D matrix of pixel values.
Color Channels:
  - Grayscale image has 1 channel per pixel. For a pixel color between (0-255)
    [ 125 124  15 155
      145 145  245 87]
  - RGB image has 3 channels per pixel. For a pixel (0-255, 0-255, 0-255)
    [ (125, 145, 147) ....
      ...................]
"""
# Preprocessing pipeline
from torchvision import transforms,datasets

"""
Localized normalization:
   image is normalized with its own mean and std
Globalized Normalization:
   image is normalized with all the images mean and std
"""

transform=transforms.Compose([
    transforms.ToTensor(),  # Scales pixels values from 0-255 to 0-1
    transforms.Normalize(mean=(0.1307,),std=(0.3081))  # Globalized values for MNIST, shifts the data so that mean becomes 0 and std becomes 1 overall
])


In [4]:
# Download dataset
train_dataset=datasets.MNIST(
    root='data',
    train=True,
    download=True,
    transform=transform

)
test_dataset=datasets.MNIST(
    root='data',
    train=False,
    download=True,
    transform=transform
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 493kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.62MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.82MB/s]


In [5]:
"""
MNIST: Hand written digit recognition dataset
Trainset contains 60,000 images
Testset contains 10,000 images
Each image is 28x28 in shape
So there is 784 pixels for each image

We are given 785 columns for each image
X: column 1-784 (pixels)
y: column 785 (digits)

Model training steps in each epoch:
   1. Forward propagation / Predicts the outcome given the input values / logits
   2. Calculate the loss
   3. Calculate Gradient w.r.t weights using backpropagation
   4. Update weights

layer 1: w1
layer 2: y = f(w1)
layer 3: z = f(w2)
loss = loss_fn(z, actual_z)

We need to know how much each weight (w1, w2) contributed to the error.

dJ
___
dw2

dJ      dJ       dw2
___ = ______ x _____
dw1     dw2      dw1

Batch
=======
We have 60000 training images
We want 10 epochs to run for training

Gradient Descent (GD) is an optimization algorithm that minimizes a model's cost function by updating parameters iteratively.
Batch GD uses the entire dataset, offering precise convergence but slow speed.
Stochastic GD (SGD) uses one example per update, allowing fast, frequent updates that are noisy.
Mini-Batch GD balances these, using small data subsets for faster, stabler convergence, and is the standard approach in deep learning. Batch size 32, 64, 128, 16, 8, 4


for epoch in epochs:
    train_batches = randomly distribute the training samples into batches
    test_batches = randomly distribute the test samples into batches
    for train_batch in train_batches:
        1. Predict the outcome given train_batch inputs
        2. Calculate loss
        3. Calculate gradients w.r.t weights
        4. Update weights
        5. Validate the performance on the test_batches

A batch size is a random sample from the training set.
batch size 32, 64, 128
"""
from torch.utils.data import DataLoader

train_loader=DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader=DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [6]:
class Perceptron(nn.Module):
  def __init__(self,input_size):
    super(Perceptron,self).__init__()
    self.w=nn.Parameter(torch.randn(input_size))
    self.b=nn.Parameter(torch.randn(1))

  def forward(self,x):
    x=x @ self.w+self.b
    return x

In [7]:
input_size=28*28
sample_input=torch.randn(input_size)
sample_input

tensor([ 0.2329, -0.0356,  0.5796, -1.0294,  1.1207,  1.0436, -0.4120,  0.0826,
         0.0820, -0.0669,  1.2531, -0.1799,  0.6627, -0.1732,  0.9173, -0.4364,
        -0.8443, -0.1002,  1.1008, -0.0785, -0.4062, -1.9043, -0.3758,  0.6975,
        -0.2557, -0.3822, -2.2710, -0.0243,  0.3159,  0.0494, -0.2075, -0.2686,
         0.4302, -0.4279,  1.0752,  0.6817, -0.8676,  1.1914,  0.7909, -0.0738,
        -0.1782,  0.7049,  0.3644, -0.0308,  0.0969,  1.4403, -1.8638, -0.2298,
         0.8571, -0.5830, -0.3066, -1.3204,  0.4840,  2.3164,  0.0724, -0.4619,
        -0.2789, -1.4704,  1.0421, -0.6439, -0.0372,  0.0788, -1.2215,  1.0876,
        -0.9279, -0.0407, -0.1720,  0.6477, -0.3406, -1.7042, -1.3252,  1.0368,
        -0.8124, -0.6976, -2.4770,  0.6338, -1.2541,  0.6449,  2.5381,  1.5168,
        -0.4613, -0.9510, -0.7020,  0.4678, -0.1149, -0.2277,  2.2701,  0.2306,
         0.0230,  2.1419, -1.2068,  0.2155,  0.1903,  0.3853, -0.2563, -0.1447,
        -1.4724,  0.1080,  0.6196,  0.45

In [8]:
model=Perceptron(input_size)
output=model(sample_input)
print(output)

tensor([0.9765], grad_fn=<AddBackward0>)


In [12]:
class ReLU(nn.Module):
  def __init__(self):
    super(ReLU,self).__init__()

  def forward(self,x):
    return torch.maximum(torch.tensor(0.0),x)



In [13]:
relu=ReLU()
output=relu(output)
print(output)

tensor([0.9765], grad_fn=<MaximumBackward0>)
